# Assignment: Extend the az.ipynb Lab

**Based on:** `Lab2.ipynb` (the Module 3 lab).

This week's assignment is short on purpose: take your working `Lab2.ipynb` lab and add **one
more step** to the chain. No new concepts, no new setup, no new libraries — just one more
chained LLM call that builds on what you already have.

**Two deployments this time:** `gpt-5.1-ptu` is the default deployment for every existing
step (Steps 1–4). The new step you add (Step 5) must call `gpt-5.4-ptu` instead.

## Step 1 — Start from your working lab

- Make a copy of your completed `Lab2.ipynb` (e.g. rename the copy `assignment3.ipynb`), or
  continue directly inside this notebook — either is fine.
- Copy in your working code from the lab's Steps 1–4: the imports and `.env` config, the
  `AzureOpenAI` client, the `chat()` helper, and the chain itself (fun fact → generate a hard
  question → answer it → evaluate the answer).
- Confirm your `.env`'s `AZURE_APIM_OPENAI_DEPLOYMENT` is set to `gpt-5.1-ptu` — this stays
  the default deployment for Steps 1–4, unchanged.

Run those cells first and confirm they still work before moving on.

In [1]:
# TODO: paste your working Lab2.ipynb code here (Steps 1-4):
from dotenv import load_dotenv
import os
import sys

from openai import AzureOpenAI
#   - imports + load_dotenv + config variables (AZURE_APIM_OPENAI_DEPLOYMENT = "gpt-5.1-ptu")
# Read .env and override any existing process env values.
load_dotenv(override=True)


# APIM settings from .env. Endpoint must be the host only, e.g.
# https://apim-azr-ue2-bgpt-prd-ucin.azure-api.net  (no /openai/deployments)
api_key = os.getenv("AZURE_APIM_OPENAI_SUBSCRIPTION_KEY")
api_version = os.getenv("AZURE_APIM_OPENAI_API_VERSION")
endpoint = os.getenv("AZURE_APIM_OPENAI_ENDPOINT")
deployment = os.getenv("AZURE_APIM_OPENAI_DEPLOYMENT")
deployment_2 = os.getenv("AZURE_APIM_OPENAI_DEPLOYMENT_2")
#   - the AzureOpenAI client
if not all([api_key, api_version, endpoint, deployment]):
    sys.exit(
        "Missing Azure APIM settings. Set AZURE_APIM_OPENAI_SUBSCRIPTION_KEY, "
        "AZURE_APIM_OPENAI_API_VERSION, AZURE_APIM_OPENAI_ENDPOINT, and "
        "AZURE_APIM_OPENAI_DEPLOYMENT in your .env file."
    )

print(f"Azure APIM key exists and begins {api_key[:8]}")
print(f"Deployment: {deployment}")
print(f"Deployment 2: {deployment_2}")
#   - the chat() helper
openai = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint,
)
# On Azure the `model` argument is the *deployment name*, not an OpenAI model id.
# GPT-5 deployments need max_completion_tokens (max_tokens is rejected).
# Sync client pointed at Azure APIM.
#       NOTE: give chat() an optional `deployment` parameter that defaults to the
#       module-level `deployment` variable, so a single call can override it later:
def chat(messages, max_completion_tokens=5000, deployment=deployment):
                return openai.chat.completions.create(
                    model=deployment, messages=messages,
                    max_completion_tokens=max_completion_tokens,
                )
#   - the chain: fact -> question -> answer -> evaluation (all using the default deployment)
# 1) Step 1
messages = [{"role": "user", "content": "Tell me a short fun fact"}]
response = chat(messages)
print(response.choices[0].message.content)

# 2) Ask the model to invent a hard IQ-style question
question = (
    "Please propose a hard, challenging question to assess someone's IQ. "
    "Respond only with the question."
)
messages = [{"role": "user", "content": question}]
response = chat(messages)
question = response.choices[0].message.content
print(question)

# 3) Ask the model to answer that question
messages = [{"role": "user", "content": question}]
response = chat(messages)
answer = response.choices[0].message.content
print(answer)

# 4) Ask the model to evaluate the answer
deployment="gpt-5.4-ptu"
message = f"""
Here is a question:
{question}

And here is a possible answer that might be correct or incorrect:
{answer}

Please evaluate if the answer is correct or incorrect.
"""
print(message)


Azure APIM key exists and begins 20efe18c
Deployment: gpt-5.1-ptu
Deployment 2: gpt-5.4-ptu
Octopuses have three hearts: two pump blood to the gills, and one pumps it to the rest of the body. When they swim, the main heart actually stops beating, which is one reason they prefer to crawl instead of swim.
You are given two identical-looking coins: one is a fair coin, and the other is a biased coin that lands heads with probability 3/4. You may flip whichever coin you choose as many times as you like, observing the results but not altering the coins. Design a strategy that, with probability at least 0.99, correctly identifies which coin is the biased one using the fewest expected number of flips possible. Clearly describe the strategy and justify why it achieves the required success probability with minimal expected flips.
Strategy outline:

1. Label the coins A and B.
2. Flip each coin the same number of times, compare the number of heads, and decide based on which has more.
3. Choose th

## Step 2 — Add one more chained step

Add a **5th step** to the chain. Its prompt must be built from at least one variable you
already have (`fact`, `question`, `answer`, or the evaluation text) — the same chaining
pattern as every other step in the lab.

**This step must call `gpt-5.4-ptu`, not the default deployment.** Pass it explicitly when
you call `chat()`:

```python
response = chat(messages, deployment="gpt-5.4-ptu")
```

Pick **one** idea below, or invent your own:

- Rate the difficulty of the question on a 1–10 scale, with a one-sentence justification.
- Rewrite the answer in one simple sentence a 10-year-old could understand.
- Suggest one new, related fun fact that connects to the original topic.
- Translate the final answer into a language of your choice.
- Write a one-line verdict on whether the model's own answer was actually correct, and why.

Store the result in its own variable, and print it clearly labeled (e.g. `=== STEP 5
(gpt-5.4-ptu) ===`).

In [2]:
fun_fact = ("Tell me a short fun fact")
messages = [{"role": "user", "content": fun_fact}]
response = chat(messages, deployment="gpt-5.1-ptu")
fun_fact = response.choices[0].message.content
print(fun_fact)


Octopuses have three hearts and blue blood—and when they’re stressed, they can squirt ink that actually dulls a predator’s sense of smell.


In [3]:
# TODO: Step 5 - build a new prompt using an earlier variable
#   (fact, question, answer, and/or the evaluation)
# TODO: call chat(messages, deployment="gpt-5.4-ptu") -- do NOT use the default deployment here
# TODO: extract the result, and print it clearly labeled
rate_fact = (f"Rate on a scale from 1-10 on how cool is the {fun_fact}. Give a one-sentence justification, then give a cooler fun fact")
messages = [{"role": "user", "content": rate_fact}]
step5 = chat(messages, deployment="gpt-5.4-ptu")
step5_text = step5.choices[0].message.content
print("=== STEP 5 (gpt-5.4-ptu) ===")
print(step5_text)


=== STEP 5 (gpt-5.4-ptu) ===
**9/10** — It’s wildly cool because octopuses already sound alien, and having **three hearts, blue blood, and “sensory-jamming” ink** makes them feel like real-life sci‑fi creatures.

**Cooler fun fact:** **Mantis shrimp can throw a punch so fast it creates tiny underwater shockwaves that briefly reach temperatures close to the surface of the sun.**


## Reflection

Answer in a sentence or two each:

1. **Which earlier variable(s) did your Step 5 prompt use, and why that one?**
2. **What would break if you ran Step 5 before the step it depends on?**
3. **Why might a real project deliberately use a different deployment (e.g. a stronger or
   more expensive model) for just one step in a chain, instead of using it everywhere?**

### My reflection

1. I used the fun fact variable, because I wanted to see AI's rating of the fun facts and if it could think of a better one. I also thought you areadt did an example on a complex question so I decided to experiment with something else.
2. Error. Because I created a variable for the fun fact (fun_fact), and I feed it into step 5, step 5 needs the variable in order to run. 
3. Because in example case of a step chain where a code generated by one model gets checked, you would want a more expensive, less hallucinating model checking/reviewing it for approval. Students in class don't grade eachother's work, the teacher or TA does.

## Submission checklist

- [x] Notebook runs top to bottom without errors (`Kernel → Restart & Run All`)
- [x] `.env` file is **not** included in your submission
- [x] Step 5 is clearly labeled and its prompt uses at least one earlier variable
- [x] Reflection questions are answered